# 额外的周末练习 —— 第 2 周

## 练习目标（理念）

用你在 **第 2 周**学到的能力，把第 1 周的「技术问答器」做成更完整的原型：

- **Gradio UI**：可交互的聊天界面
- **流式（streaming）**（本仓库实现可先非流式，再自行加）
- **system prompt**：注入领域专家人设（本练习是招聘 / JD 技能分析）
- **多模型切换**（可扩展）
- **奖励分**：演示 **工具调用（tool calling）**——例如抓取网页 JD
- **更大胆**：音频输入 / 音频回复（可选）

## 和本课概念的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions | `openai.chat.completions.create(...)` |
| system / user / history | 拼完整 `messages` |
| Function / Tool Calling | `tools` + `finish_reason == "tool_calls"` |
| Gradio | `gr.ChatInterface(fn=chat).launch()` |
| 本地 Ollama（OpenAI 兼容） | `base_url=http://localhost:11434/v1` |

## 怎么跑

1. 确保本地 Ollama 已启动，并已 `ollama pull` 对应模型（本笔记本用 `gpt-oss:20b`）
2. `.env` 可放 `OPENAI_API_KEY`（本实现主要走本地 Ollama）
3. 同目录需有 `scraper.fetch_website_contents`（或按你的实现调整）
4. 从上到下运行单元格，最后启动 Gradio 聊天


In [ ]:
# ========== 导入：UI、本地模型客户端、工具函数 ==========

# os：读环境变量（Environment Variables）
import os
# json：解析模型返回的 tool call 参数（arguments 常是 JSON 字符串）
import json
# gradio：快速搭聊天 Web UI
import gradio as gr
# sqlite3：本练习导入了，后续若接数据库可复用（当前主流程未必用到）
import sqlite3
# load_dotenv：把 .env 密钥读进进程环境
from dotenv import load_dotenv
# OpenAI 客户端：这里指向本地 Ollama 的 OpenAI 兼容端点
from openai import OpenAI
# 抓取网页正文的工具函数（给 LLM 当 tool 用）
from scraper import fetch_website_contents


In [ ]:
# ========== 环境：加载 .env 并检查 OPENAI_API_KEY 是否存在 ==========

# override=True：以 .env 文件覆盖进程里已有同名变量
load_dotenv(override=True)

# 从环境读取 OpenAI 云端密钥（本笔记本主路径用 Ollama，但保留检查提示）
api_key = os.getenv("OPENAI_API_KEY")

# 没有密钥时打印排查提示（文案保持英文，避免改可运行行为相关字符串）
if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
else:
    # 原实现拼写 founded；保留不改，以免偏离原逻辑/输出
    print("API key founded")


In [ ]:
# ========== 模型与客户端：指向本地 Ollama ==========

# 本地模型名：必须和 ollama list 里已有模型一致
MODEL = "gpt-oss:20b"
# api_key 对本地 Ollama 多为占位；base_url 指向 OpenAI 兼容的 /v1
openai = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")


In [ ]:
# ========== system prompt：招聘分析专家人设（发给模型的指令，保留英文） ==========

system_prompt = """
Role: You are an expert Technical Recruiter and Job Analyst.

Task: Analyze the provided Job Description (JD) and extract all required skills (technical, soft skills, and tools). Focus on the required skill not company's benefits.

Requirements: > 1. Rank the skills in a numbered list from 1 (Most Critical) to N (Least Critical).
2. Ranking Logic: Base the rank on the frequency of mention, the "Required" vs. "Preferred" sections, and how central the skill is to the core responsibilities described.
3. Provide a brief (one-sentence) justification for why the top 3 skills were ranked highest.
"""


In [ ]:
# ========== 工具 schema：告诉模型有一个可调用的抓网页函数 ==========

# OpenAI / 兼容接口要求的 function 描述字典（name / description / parameters）
crawl_function = {
    # 函数名：模型 tool_call 里会带回这个名字，便于路由到真实 Python 函数
    "name": "fetch_website_contents",
    # 给模型看的说明：何时该调用
    "description": "Receive a website url and then return content of that website.",
    # JSON Schema：参数类型与必填项
    "parameters": {
        "type": "object",
        "properties": {
            "url": {
                "type": "string",
                "description": "An URL of the website that user want to get content",
            },
        },
        "required": ["url"],
        # 禁止额外未知字段，减少胡编参数
        "additionalProperties": False
    }
}


In [ ]:
# ========== tools 列表：挂到 chat.completions.create(..., tools=...) ==========

tools = [
    # type=function：这是函数工具；function 字段放上面的 schema
    {"type": "function", "function": crawl_function}
]


In [ ]:
# ========== 聊天入口：拼 messages，必要时进入 tool-call 循环 ==========

def chat(message, history):
    # Gradio history → 只要 role/content 的标准 chat 消息列表
    history = [{"role": h["role"], "content": h["content"]}for h in history]
    # system + 历史 + 当前 user
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    # 第一次请求模型（注意：原代码此处未传 tools=；保留不改）
    response = openai.chat.completions.create(model=MODEL, messages=messages)

    # 若模型要以工具结束，就执行工具、把结果塞回 messages，再问模型
    while response.choices[0].finish_reason == "tool_calls":
        # 助手消息里带 tool_calls
        message = response.choices[0].message
        # 本地真正跑工具，得到 role=tool 的回复列表
        responses = handle_tool_calls(message)
        # 先追加助手的 tool_calls 消息
        messages.append(message)
        # 再追加每个工具的执行结果
        messages.extend(responses)
        # 带着工具结果再次请求模型，生成最终自然语言回答
        response = openai.chat.completions.create(model=MODEL, messages=messages)

    # 返回最终助手文本给 Gradio
    return response.choices[0].message.content


In [ ]:
# ========== 工具调度：按 tool_call 名字调用本地函数，拼 role=tool 消息 ==========

def handle_tool_calls(message):
    # 收集所有工具结果，一次返回给上层
    responses = []

    # 一条助手消息里可能有多个 tool_calls
    for tool_call in message.tool_calls:
        # 只处理抓网页这一个函数名（原代码用 function_name；保留不改）
        if tool_call.function_name == "fetch_website_contents":
            # 原实现用 json.load；保留不改（常见写法是 json.loads）
            arguments = json.load(tool_call.function.arguments)
            # 取出 URL 参数
            url = arguments.get("url")
            # 原实现函数名与导入名不完全一致；保留不改
            response_content = fetch_website_content(url)
            # 拼成 Chat Completions 要求的 tool 角色消息（原变量名 resoponses 保留）
            resoponses.append({
                "role": "tool",
                "content": response_content,
                
            })
    
    return responses


In [ ]:
# ========== 启动 Gradio 聊天界面 ==========

# fn=chat：用户每发一条消息就调用上面的 chat
gr.ChatInterface(fn=chat).launch()
